## part def and attributes

This notebook introduces `part def` with typed attributes; after running it you can define component types with numeric parameters.

The previous notebook established `ToastingSystem` as the abstract system concept. This notebook introduces concrete component definitions: `Heater` carries a `power` attribute, and `HeatingSystem` and `ControlSystem` are named as distinct subsystem types, with no hierarchy yet.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch01-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch01-cumulative.sysml` file declares the four foundational part definitions: `ToastingSystem` (abstract system concept with a doc annotation), `Heater` (with a `power : Real` attribute), `HeatingSystem` and `ControlSystem` (specializations via `:>`), and `Toaster` (composed from `heating` and `control` parts). These constructs form the structural skeleton that every later chapter extends.

In [ ]:
# Negative control: attribute type must resolve to a known classifier.
# Referencing an undefined type causes an "unresolved reference" error.
bad_source = """
package Bad {
    part def Heater {
        attribute power : UnknownType default = 800.0;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
heater = model.find("ToasterDemo::Heater")
assert heater is not None
attrs = heater.attributes()
print(f"Heater attributes ({len(attrs)}):")
for a in attrs:
    print(f"  {a.id}")

print()
for e in model.query():
    d = e.as_dict()
    if d["@type"] == "PartDefinition":
        print(f"PartDefinition: {d['qualifiedName']}")
conn.close()

`part def Heater { attribute power : Real default = 800.0; }` is the A-F declaration; OpenSysML resolves `Real` from the imported library and stores the attribute (O-S); `heater.attributes()` returns the attribute symbol (E).

Try the chapter exercise in `exercises/ch01/exercise.ipynb`: define a `BrewUnit` part def with a `brewTemp` attribute and verify it loads.